# Connect to pawsey0411

In [ ]:
import socket
socket.gethostbyname("projects.pawsey.org.au")
socket.create_connection(("projects.pawsey.org.au", 443), timeout=10).close()
print("Endpoint reachable")

ENDPOINT = "https://projects.pawsey.org.au"
BUCKET = "weather"
SCOPE = "pawsey0411"

ACCESS_KEY = dbutils.secrets.get(scope=SCOPE, key="access_key")
SECRET_KEY = dbutils.secrets.get(scope=SCOPE, key="secret_key")

In [ ]:
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name="us-east-1",
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)

# Print visible buckets
try:
    print("Buckets visible:", [b["Name"] for b in s3.list_buckets().get("Buckets", [])])
except ClientError as e:
    print("list_buckets not allowed:", e.response.get("Error", {}))

# Equivalent to rclone lsd pawsey0411:weather
resp = s3.list_objects_v2(Bucket=BUCKET, Delimiter="/")
prefixes = [p["Prefix"] for p in resp.get("CommonPrefixes", [])]
print("Top-level pseudo-directories:", prefixes)

# Top-level objects sample
root_objects = [o["Key"] for o in resp.get("Contents", [])]
print("Top-level objects sample:", root_objects[:20])